In [15]:
import pandas as pd
import numpy as np
import pycountry_convert as pc

from bokeh.plotting import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Set2

output_notebook()

Loading BokehJS ...

In [16]:
df = pd.read_parquet("../../results/country_confustion_matrix.parquet")

In [17]:
df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)

In [19]:
def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except Exception as e:
        return None  # or handle the exception/log as needed

In [20]:
df["continent"] = df["country"].apply(country_to_continent)
df = df[~df.continent.isna()]

In [21]:
hdi_df = pd.read_csv("../../data/human_development_index.csv")[["iso3", "hdi_2020"]]

In [22]:
df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

In [23]:
df.head(5)

,country,gid,pixel_count,TN,FP,FN,TP,f1,continent,hdi
0,Zimbabwe,ZWE,1928357,1818452,1382,101555,6968,0.119241,Africa,0.600
1,Zambia,ZMB,3614517,3522670,2705,81554,7588,0.152622,Africa,0.570
2,South Africa,ZAF,6513508,6150356,56330,175983,130839,0.529722,Africa,0.727
3,Yemen,YEM,2198652,2170523,10510,11706,5913,0.347394,Asia,0.460
4,Samoa,WSM,13745,12100,108,1332,205,0.221622,Oceania,0.712


In [24]:



df['size'] = np.log1p(df['pixel_count'])

In [25]:
continents = df['continent'].unique().tolist()
palette = Set2[max(3, len(continents))]

source = ColumnDataSource.from_df(df)


fig = figure(
    x_axis_label='Human Development Index', 
    y_axis_label='F1',
    width=980
)
fig.scatter(
    x="hdi",
    y="f1", 
    source=source,
    alpha=0.8,
    color=factor_cmap('continent', palette=palette, factors=continents),
    legend_field='continent',
    size="size"
)

hover = HoverTool(tooltips=[
    ("Country", "@country"),
    ("F1", "@f1"),
    ("Pixel Count", "@pixel_count"),
])

fig.legend.location = "bottom_right"

fig.add_tools(hover)
show(fig)

*size of the circles is `log(total pixel count)`